# Formula 1 SQL Case Study — *What Makes a Champion?*

This notebook is the presentation layer for the MySQL project in this folder. It reconnects to the `f1_analytics` database, re-runs the thesis queries, and turns their results into reviewer-friendly tables and charts.

The guiding question is: **what separates a World Champion from the rest of the grid—peak pace, consistency, reliability, the car, or a combination?**

The notebook deliberately keeps computation in SQL. Python is used for connection management, tabular display, and visualisation. The authoritative SQL definitions are in `10_capstone_thesis.sql`; the queries below mirror those definitions so the narrative and the database remain aligned.

## Before running

1. Build the database by running `run_all.sql` from the project `sql/` directory.
2. Install the packages in `requirements.txt`.
3. Set `F1_DB_HOST`, `F1_DB_PORT`, `F1_DB_NAME`, `F1_DB_USER`, and `F1_DB_PASSWORD` in the environment.
4. Run all cells from top to bottom.

The notebook saves charts to `results/screenshots/`. It does not store a password in the notebook source.

In [ ]:
import os
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import pandas as pd
from sqlalchemy import URL, create_engine, text

# Locate the project even when Jupyter was launched from either sql/ or sql/notebook/.
candidates = [Path.cwd(), Path.cwd().parent, Path(__file__).resolve().parent.parent if '__file__' in globals() else None]
SQL_DIR = next((p for p in candidates if p and (p / '01_schema.sql').exists()), None)
if SQL_DIR is None:
    raise FileNotFoundError('Run this notebook from the project sql/ directory or its notebook/ subdirectory.')

SHOTS = SQL_DIR / 'results' / 'screenshots'
SHOTS.mkdir(parents=True, exist_ok=True)

mpl.rcParams.update({
    'figure.dpi': 110,
    'axes.grid': True,
    'grid.alpha': 0.25,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 11,
    'axes.titleweight': 'bold',
})
F1_RED = '#e10600'

db_password = os.getenv('F1_DB_PASSWORD')
if db_password is None:
    raise RuntimeError('Set F1_DB_PASSWORD before running the notebook.')

db_url = URL.create(
    'mysql+pymysql',
    username=os.getenv('F1_DB_USER', 'root'),
    password=db_password,
    host=os.getenv('F1_DB_HOST', '127.0.0.1'),
    port=int(os.getenv('F1_DB_PORT', '3306')),
    database=os.getenv('F1_DB_NAME', 'f1_analytics'),
    query={'charset': 'utf8mb4'},
)
engine = create_engine(db_url)

def q(sql: str) -> pd.DataFrame:
    with engine.connect() as conn:
        return pd.read_sql_query(text(sql), conn)

def save_chart(fig, filename: str) -> None:
    fig.tight_layout()
    fig.savefig(SHOTS / filename, bbox_inches='tight')
    plt.show()

print('Project:', SQL_DIR)
print('Database:', q('SELECT DATABASE() AS database_name, VERSION() AS mysql_version').to_dict('records')[0])

## 1. Validate the analytical foundation

The first result is a compact data-quality check. The full version lives in `03_data_profiling.sql`; this preview confirms the dimensions that the thesis depends on before any chart is made.

In [ ]:
profile = q('''
SELECT 'seasons' AS table_name, COUNT(*) AS n_rows FROM seasons
UNION ALL SELECT 'races', COUNT(*) FROM races
UNION ALL SELECT 'drivers', COUNT(*) FROM drivers
UNION ALL SELECT 'constructors', COUNT(*) FROM constructors
UNION ALL SELECT 'results', COUNT(*) FROM results
UNION ALL SELECT 'driver_standings', COUNT(*) FROM driver_standings
UNION ALL SELECT 'lap_times', COUNT(*) FROM lap_times
ORDER BY table_name
''')
display(profile)

quality = q('''
SELECT
  (SELECT MIN(year) FROM races) AS first_season,
  (SELECT MAX(year) FROM races) AS last_season,
  SUM(position IS NULL) AS unclassified_results,
  SUM(grid = 0) AS missing_or_pit_lane_grids
FROM results
''')
display(quality)

## 2. Consistency versus peak

A champion is identified as position 1 in the drivers' standings at the final race of a season. Comparing that driver with the runner-up allows us to ask whether the title is associated with more wins, fewer DNFs, or both.

In [ ]:
consistency = q('''
WITH final_race AS (
  SELECT year, raceId, ROW_NUMBER() OVER (PARTITION BY year ORDER BY round DESC) AS rr
  FROM races
),
top2 AS (
  SELECT fr.year, ds.driverId, ds.position
  FROM final_race fr
  JOIN driver_standings ds ON ds.raceId = fr.raceId
  WHERE fr.rr = 1 AND ds.position IN (1, 2)
),
season_stats AS (
  SELECT ra.year, re.driverId,
         SUM(re.positionOrder = 1) AS wins,
         SUM(re.position IS NULL) AS dnfs,
         COUNT(*) AS races
  FROM results re
  JOIN races ra ON ra.raceId = re.raceId
  GROUP BY ra.year, re.driverId
)
SELECT CASE t.position WHEN 1 THEN 'Champion' ELSE 'Runner-up' END AS finishing,
       ROUND(AVG(ss.wins), 2) AS avg_wins,
       ROUND(AVG(ss.dnfs), 2) AS avg_dnfs,
       ROUND(100 * AVG(ss.wins / NULLIF(ss.races, 0)), 1) AS avg_win_rate_pct
FROM top2 t
JOIN season_stats ss ON ss.year = t.year AND ss.driverId = t.driverId
GROUP BY t.position
ORDER BY t.position
''')
display(consistency)

fig, ax = plt.subplots(figsize=(7, 5))
x = range(len(consistency))
width = 0.38
ax.bar([i - width / 2 for i in x], consistency['avg_wins'], width, label='Average wins', color=F1_RED)
ax.bar([i + width / 2 for i in x], consistency['avg_dnfs'], width, label='Average DNFs', color='#333333')
ax.set_xticks(list(x), consistency['finishing'])
ax.set_ylabel('Average count per title season')
ax.set_title('Champions win more and retire less often')
ax.legend()
save_chart(fig, '02_consistency_vs_peak.png')

**Readout:** The recorded run shows 6.37 wins and 2.24 DNFs for champions versus 3.45 wins and 3.07 DNFs for runners-up. The result supports a combined pace-and-finishing story rather than a simple “champions are only faster” explanation.

## 3. The age curve

This query calculates age at each race and averages points per race by age. Requiring at least 100 starts at an age reduces noise from tiny samples. It is descriptive: it does not control for era, car quality, or selection into a seat.

In [ ]:
age = q('''
SELECT TIMESTAMPDIFF(YEAR, d.dob, ra.date) AS age,
       COUNT(*) AS races,
       ROUND(AVG(re.points), 3) AS avg_points_per_race
FROM results re
JOIN races ra ON ra.raceId = re.raceId
JOIN drivers d ON d.driverId = re.driverId
WHERE d.dob IS NOT NULL
GROUP BY age
HAVING races >= 100
ORDER BY age
''')
display(age)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(age['age'], age['avg_points_per_race'], marker='o', color=F1_RED)
ax.set_title('Average points per race by driver age')
ax.set_xlabel('Age at race')
ax.set_ylabel('Average points per race')
save_chart(fig, '03_age_curve.png')

**Readout:** The saved results show a broad, uneven plateau from the early twenties into the mid-thirties rather than one universal peak. This is a useful descriptive pattern, but it should not be over-read as proof that age alone determines performance.

## 4. Car versus driver

To test whether champions demonstrated success across machinery, we count the distinct constructors for which each championship-winning driver won a race.

In [ ]:
multi_team = q('''
WITH final_race AS (
  SELECT year, raceId, ROW_NUMBER() OVER (PARTITION BY year ORDER BY round DESC) AS rr
  FROM races
),
champions AS (
  SELECT DISTINCT ds.driverId
  FROM final_race fr
  JOIN driver_standings ds ON ds.raceId = fr.raceId
  WHERE fr.rr = 1 AND ds.position = 1
),
teams AS (
  SELECT ch.driverId, COUNT(DISTINCT re.constructorId) AS winning_teams
  FROM champions ch
  JOIN results re ON re.driverId = ch.driverId AND re.positionOrder = 1
  GROUP BY ch.driverId
)
SELECT SUM(winning_teams > 1) AS multi_team,
       SUM(winning_teams = 1) AS single_team
FROM teams
''')
multi = int(multi_team.loc[0, 'multi_team'])
single = int(multi_team.loc[0, 'single_team'])
display(multi_team)

fig, ax = plt.subplots(figsize=(6, 6))
ax.pie([multi, single], labels=[f'Multiple teams\n({multi})', f'Single team\n({single})'],
        autopct='%1.1f%%', colors=[F1_RED, '#cccccc'], startangle=90,
        wedgeprops={'edgecolor': 'white'})
ax.set_title('Champions who won races for more than one constructor')
save_chart(fig, '04_multi_team_champions.png')
print(f'{multi} of {multi + single} race-winning champions ({100 * multi / (multi + single):.1f}%) won with multiple teams.')

**Readout:** 26 of 34 race-winning champions in the recorded run won for more than one constructor. That supports the idea that the strongest drivers can transfer performance across teams, while the analysis still acknowledges that a competitive car is a prerequisite for many wins.

## 5. Qualifying and season points

Each point in the scatter plot is one driver-season. Lower average grid position is better. The correlation query retains zero-point driver-seasons when a valid average grid exists, matching `10_capstone_thesis.sql`.

In [ ]:
qualifying_summary = q('''
WITH driver_seasons AS (
  SELECT ra.year, re.driverId,
         AVG(NULLIF(re.grid, 0)) AS avg_grid,
         SUM(re.points) AS points
  FROM results re
  JOIN races ra ON ra.raceId = re.raceId
  GROUP BY ra.year, re.driverId
  HAVING avg_grid IS NOT NULL
)
SELECT COUNT(*) AS driver_seasons,
       ROUND((COUNT(*) * SUM(avg_grid * points) - SUM(avg_grid) * SUM(points)) /
         NULLIF(SQRT(
           (COUNT(*) * SUM(avg_grid * avg_grid) - POW(SUM(avg_grid), 2)) *
           (COUNT(*) * SUM(points * points) - POW(SUM(points), 2))
         ), 0), 4) AS pearson_r
FROM driver_seasons
''')

scatter = q('''
SELECT ra.year, AVG(NULLIF(re.grid, 0)) AS avg_grid, SUM(re.points) AS points
FROM results re
JOIN races ra ON ra.raceId = re.raceId
GROUP BY ra.year, re.driverId
HAVING avg_grid IS NOT NULL
''')
display(qualifying_summary)

r_value = float(qualifying_summary.loc[0, 'pearson_r'])
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(scatter['avg_grid'], scatter['points'], s=10, alpha=0.35, color=F1_RED)
ax.set_title(f'Average race-start grid versus season points (Pearson r = {r_value:.4f})')
ax.set_xlabel('Average grid position (lower is better)')
ax.set_ylabel('Season points')
save_chart(fig, '05_qualifying_vs_points.png')

**Readout:** The recorded result is `r = -0.4222` across 3,040 driver-seasons. The negative sign is expected because a smaller grid number is a better starting position. The magnitude is moderate: qualifying creates an advantage, but it does not determine the full season.

## 6. Reliability: mechanical DNFs

The final thesis query uses a transparent curated list of mechanical status labels. Driver-error labels are not included, so the measure is specifically mechanical-DNF rate rather than total DNF rate.

In [ ]:
reliability = q('''
WITH final_race AS (
  SELECT year, raceId, ROW_NUMBER() OVER (PARTITION BY year ORDER BY round DESC) AS rr
  FROM races
),
champions AS (
  SELECT fr.year, ds.driverId
  FROM final_race fr
  JOIN driver_standings ds ON ds.raceId = fr.raceId
  WHERE fr.rr = 1 AND ds.position = 1
),
mechanical AS (
  SELECT ra.year, re.driverId,
         SUM(s.status IN (
           'Engine','Gearbox','Transmission','Hydraulics','Electrical',
           'Suspension','Brakes','Clutch','Overheating','Power Unit',
           'Fuel system','Oil leak','Water leak','Driveshaft','Radiator',
           'Throttle','Turbo','Exhaust','Differential','Alternator',
           'Fuel pump','Ignition','Oil pressure','Wheel','Halfshaft',
           'Mechanical','Power loss','Fuel leak','Fuel pressure',
           'Cooling system','Vibrations','ERS','Battery','Distributor',
           'Pneumatics','Engine fire','Spark plugs','Wheel bearing',
           'Oil line'
         )) AS mechanical_dnfs,
         COUNT(*) AS races
  FROM results re
  JOIN races ra ON ra.raceId = re.raceId
  JOIN status s ON s.statusId = re.statusId
  GROUP BY ra.year, re.driverId
)
SELECT CASE WHEN c.driverId IS NOT NULL THEN 'Champion' ELSE 'Rest of field' END AS grp,
       SUM(m.mechanical_dnfs) AS mechanical_dnfs,
       SUM(m.races) AS races,
       ROUND(100 * SUM(m.mechanical_dnfs) / NULLIF(SUM(m.races), 0), 2) AS mechanical_dnf_pct
FROM mechanical m
LEFT JOIN champions c ON c.year = m.year AND c.driverId = m.driverId
GROUP BY grp
ORDER BY grp
''')
display(reliability)

fig, ax = plt.subplots(figsize=(6, 5))
ax.bar(reliability['grp'], reliability['mechanical_dnf_pct'], color=[F1_RED, '#999999'])
for i, value in enumerate(reliability['mechanical_dnf_pct']):
    ax.text(i, value + 0.3, f'{value:.2f}%', ha='center')
ax.set_ylabel('Percentage of race starts')
ax.set_title('Mechanical-DNF rate: champions versus the field')
save_chart(fig, '06_reliability.png')

## Verdict

The evidence points to a compound advantage. Champions win substantially more races than runners-up, but they also lose fewer starts to retirement. Better grid position is helpful without being decisive, and most race-winning champions in the snapshot have won for more than one constructor.

The most defensible one-line conclusion is:

> **A champion is fast often enough, in reliable enough machinery, to finish the races that a fast-but-fragile rival cannot.**

For the SQL implementation, compare this notebook with `10_capstone_thesis.sql`, then read `docs/findings_report.md` for the saved numerical interpretation and `README.md` for limitations and reproduction steps.